In [1]:
import fine as fn
import pyomo.environ as pyomo


# Step 1: Define the Energy System Model
esM = fn.EnergySystemModel(
    locations={"A"},
    commodities={"electricity"},
    commodityUnitsDict={"electricity": "GW"},
    materials={"steel", "copper"},
    materialUnitsDict={"steel": "tons", "copper": "kg"}
)
esM.pyM = pyomo.ConcreteModel()


In [2]:
esM.processedMaterialBalanceLimit = {
    0: {  # Investment Period 0
        "copper": {"A": 10},  
        "steel": {"A": 5},    
    }
}

In [3]:
# Step 2: Add a Material Source (Raw Material Supplier)
esM.add(
    fn.Source(
        esM=esM, 
        name="Electricity",
        #materialConsumption="steel",
        commodity="electricity",
        hasCapacityVariable=True,
        materialConsumption={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
)



In [4]:
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Supply",
        #materialConsumption="steel",
        materials="steel",
        hasCapacityVariable=True,
    )
)

In [5]:
# Step 2: Add a Material Source (Raw Material Supplier)
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Supply",
        #materialConsumption="steel",
        materials="steel",
        hasCapacityVariable=True,
    )
)


/fast/home/l-soeltzer/code/fine/fine/component.py:729: UserWarning: Component identifier Steel Supply already exists. Data will be overwritten.
  warnings.warn(


In [6]:
# Step 2: Add a Material Source (Raw Material Supplier)
esM.add(
    fn.Source(
        esM=esM, 
        name="Copper Supply",
        #materialConsumption="steel",
        materials="copper",
        hasCapacityVariable=True,
        #materialConsumption={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        #materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
)

In [7]:
# Step 4: Add a Storage Component that Requires Materials

esM.add(
    fn.Storage(
        esM=esM,
        name="Battery",
        commodity="electricity",
        chargeEfficiency=0.9,
        dischargeEfficiency=0.9,
        materialConsumption={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
) 

In [8]:
esM.add(
        fn.Sink(
            esM=esM,
            name="Electricity demand",
            commodity="electricity",
            hasCapacityVariable=False,
            operationRateFix=50,
        )
    )

In [9]:
esM.optimize()

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.5951 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(1.3655 sec)

Declaring shared potential constraint...
		(0.0003 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.3460 sec)

		(0.0000 sec)

Declaring objective function...
		(3.7557 sec)

Either solver not selected or specified solver not available.gurobi is set as solver.
Set parameter ServerPassword
Set parameter TSPort to value 41955
Set parameter TokenServer to value "iek3079"
Read LP format model from file /tmp/tmpkbzielah.pyomo.lp
Reading time = 0.14 seconds
x1: 78854 rows, 61338 columns, 183988 nonzeros
Set parameter QCPDual to value 1
Set parameter Threads to value 3
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (linux64 - "Rocky Linux 8

In [10]:
esM.getOptimizationSummary("SourceSinkModel", outputLevel=2)

A
Component          Property      Unit              
Electricity        capacity      [GW]          50.0
                   commissioning [GW]          50.0
                   operation     [GW*h/a]  438000.0
                                 [GW*h]    438000.0
Electricity demand operation     [GW*h/a]  438000.0
                                 [GW*h]    438000.0